# Session 4
- intro to pygame

In [1]:
pip install pygame opencv-python numpy

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [13]:
import cv2
import numpy as np
from ugot import ugot
got = ugot.UGOT()
got.initialize("10.66.154.25")
ugot_speed = 50
ugot_turn_speed = 60
got.open_camera()

import pygame
pygame.init()
screen = pygame.display.set_mode((640, 500)) # w, h
pygame.display.set_caption("hello") # window title
x = 400
y = 300
speed = 1
running = True
while running:
    frame = got.read_camera_data()
    if not frame:
        continue
    nparr = np.frombuffer(frame, np.uint8)
    data = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    frame_rgb = cv2.cvtColor(data, cv2.COLOR_BGR2RGB)
    surfaace = pygame.image.frombuffer(frame_rgb.tobytes(), 
                (640, 480), "RGB")
    screen.blit(surfaace, (0, 0))
    # go through event queue
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
    # check for key presses
    keys = pygame.key.get_pressed()
    if keys[pygame.K_LEFT]:
        x -= speed
    if keys[pygame.K_RIGHT]:
        x += speed
    if keys[pygame.K_UP]:
        y -= speed
    if keys[pygame.K_DOWN]:
        y += speed

    # UGOT movement
    ugot_x, ugot_y, ugot_z = 0, 0, 0
    if keys[pygame.K_a]:
        ugot_x = -ugot_speed
    if keys[pygame.K_d]:
        ugot_x = ugot_speed
    if keys[pygame.K_s]:
        ugot_y = -ugot_speed
    if keys[pygame.K_w]:
        ugot_y = ugot_speed
    if keys[pygame.K_e]:
        ugot_z = -ugot_turn_speed
    if keys[pygame.K_q]:
        ugot_z = ugot_turn_speed
    got.mecanum_move_xyz(ugot_x, ugot_y, ugot_z)

    # screen.fill((255, 255, 255)) # white background
    # # circle - surface, color, position, radius
    # pygame.draw.circle(screen, (0, 0, 255), (x, y), 50)
    pygame.display.flip() # update display
pygame.quit()

10.66.154.25:50051


In [22]:
import random
from ugot import ugot
got = ugot.UGOT()
got.initialize("10.66.154.25")
import pygame
WIDTH, HEIGHT = 600, 400
screen = pygame.display.set_mode((WIDTH, HEIGHT))

x = WIDTH // 2 # floor division
y = HEIGHT // 2
enemy_x, enemy_y = 200, 200
enemy_direction_x = random.choice([-1, 1])
enemy_direction_y = random.choice([-1, 1])
speed = 1

def clamp(value, low, high):
    return max(low, min(high, value))

def read_gyro():
    data = got.read_gyro_data()
    pitch = data[0]
    roll = data[1]
    yaw = data[2]
    # print(pitch, roll, yaw)
    return pitch, roll, yaw

center_pitch, center_roll, center_yaw = read_gyro()
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_SPACE:
                center_pitch, center_roll, center_yaw = read_gyro()
                x, y = WIDTH //2, HEIGHT // 2
    pitch, roll, yaw = read_gyro()
    # player movement
    move_x = (roll - center_roll) / 20
    move_y = (pitch - center_pitch) / -20

    x += move_x * speed
    y += move_y * speed
    # prevent rectangle from going out of window
    x = clamp(x, 0, WIDTH - 40)
    y = clamp(y, 0, HEIGHT - 60)
    
    # enemy movement
    enemy_x += enemy_direction_x * 2
    enemy_y += enemy_direction_y * 2
    if enemy_x <= 0 or enemy_x >= WIDTH - 100:
        enemy_direction_x *= -1
        # enemy_direction_x = enemy_direction_x * -1
    if enemy_y <= 0 or enemy_y >= HEIGHT - 100:
        enemy_direction_y *= -1

    screen.fill((255, 255, 255))
    player_rect = pygame.Rect(x, y, 40, 60)
    enemy_rect = pygame.Rect(enemy_x, enemy_y, 100, 100)
    pygame.draw.rect(screen, (0, 30, 255), enemy_rect)
    collision = player_rect.colliderect(enemy_rect)
    if collision:
        pygame.draw.rect(screen, (200, 30, 30), player_rect)
    else:
        pygame.draw.rect(screen, (0, 255, 30), player_rect)

    pygame.display.flip()
pygame.quit()

10.66.154.25:50051


In [ ]:
distance = got.